# Assignment #1 -- Question 2: English-to-Urdu NMT (Vanilla RNN)
Colab runner. Code lives in the GitHub repo and is cloned below. Persistent state (dataset, manifests, vocabularies, checkpoints, results) lives on Google Drive under `PROJECT_DIR` so it survives runtime disconnects.

## 0. Setup: mount Drive, clone the repo, install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/generative_ai_assignment1'
os.makedirs(PROJECT_DIR, exist_ok=True)

from pathlib import Path
NOTES_PATH = Path(PROJECT_DIR) / 'report_notes.md'

In [ ]:
from getpass import getpass

REPO_DIR = '/content/repo'
REPO_URL = 'github.com/i222502-school/generative_ai_assignment1.git'

if not os.path.exists(REPO_DIR):
    token = getpass('GitHub PAT (repo scope; only kept in this runtime): ')
    !git clone https://{token}@{REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

In [ ]:
!pip install -q -r requirements.txt

## 1. Kaggle dataset download
Needs `kaggle.json` from https://www.kaggle.com/settings -> API -> Create New Token.

**Note:** `q2_nmt_rnn/src/data.py::load_pairs` guesses the file shape (a single CSV with English/Urdu columns, or two aligned .txt files). If this dataset ships some other way, the error message here will say so -- fix `load_pairs` accordingly, it's one small function.

In [ ]:
import pathlib, shutil

kaggle_dir = pathlib.Path('/root/.kaggle')
kaggle_dir.mkdir(exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'

if not kaggle_json.exists():
    from google.colab import files
    uploaded = files.upload()  # select kaggle.json
    shutil.move(next(iter(uploaded)), kaggle_json)
kaggle_json.chmod(0o600)

import kagglehub
dataset_path = kagglehub.dataset_download('muhammadnoman76/translation-dataset')
print(dataset_path)
!find {dataset_path} -maxdepth 2

## 2. Preprocessing + reproducible split (Tasks 1-2)

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from pathlib import Path
import pandas as pd
from q2_nmt_rnn.src.data import build_manifests
from q2_nmt_rnn.src.config import SplitRatios

SEED = 42
MANIFEST_DIR = Path(PROJECT_DIR) / 'manifests'

if (MANIFEST_DIR / 'train.csv').exists():
    splits = {name: pd.read_csv(MANIFEST_DIR / f'{name}.csv') for name in ('train', 'val', 'test')}
else:
    splits = build_manifests(Path(dataset_path), manifest_dir=MANIFEST_DIR, ratios=SplitRatios(), seed=SEED)

{name: len(part) for name, part in splits.items()}

from q2_nmt_rnn.src.report_notes import append_section
append_section(
    NOTES_PATH, 'Dataset and split',
    '\n'.join(f'{name}: {len(part)} pairs' for name, part in splits.items()),
)

## 3. Tokenization and vocabularies (Task 3)

In [ ]:
from q2_nmt_rnn.src.tokenizer import tokenize
from q2_nmt_rnn.src.vocab import Vocabulary

src_vocab_path = Path(PROJECT_DIR) / 'src_vocab.json'
tgt_vocab_path = Path(PROJECT_DIR) / 'tgt_vocab.json'

if src_vocab_path.exists():
    src_vocab = Vocabulary.load(src_vocab_path)
    tgt_vocab = Vocabulary.load(tgt_vocab_path)
else:
    src_vocab = Vocabulary.build([tokenize(t) for t in splits['train']['en']])
    tgt_vocab = Vocabulary.build([tokenize(t) for t in splits['train']['ur']])
    src_vocab.save(src_vocab_path)
    tgt_vocab.save(tgt_vocab_path)

print('English vocab size:', len(src_vocab))
print('Urdu vocab size:', len(tgt_vocab))

append_section(
    NOTES_PATH, 'Vocabularies',
    f'English vocab size: {len(src_vocab)}\n\nUrdu vocab size: {len(tgt_vocab)}',
)

## 4. Sequence encoding, padding, masking, batching (Task 5)

In [ ]:
import torch
from torch.utils.data import DataLoader
from q2_nmt_rnn.src.dataset import TranslationDataset, collate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

BATCH_SIZE = 64
train_loader = DataLoader(
    TranslationDataset(splits['train'], src_vocab, tgt_vocab), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate
)
val_loader = DataLoader(
    TranslationDataset(splits['val'], src_vocab, tgt_vocab), batch_size=BATCH_SIZE, collate_fn=collate
)
test_loader = DataLoader(
    TranslationDataset(splits['test'], src_vocab, tgt_vocab), batch_size=BATCH_SIZE, collate_fn=collate
)

sample_batch = next(iter(train_loader))
{k: tuple(v.shape) for k, v in sample_batch.items()}

## 5. Primary model: vanilla RNN encoder-decoder + layer table (Task 4)

In [ ]:
from q2_nmt_rnn.src.models.seq2seq import Seq2Seq
from q2_nmt_rnn.src.models.summary import layer_table, param_counts

model = Seq2Seq(len(src_vocab), len(tgt_vocab), embed_dim=256, hidden_dim=512, dropout=0.3)
print(param_counts(model))

table = layer_table(model, sample_batch['src'][:1], sample_batch['tgt_in'][:1])
table.to_csv(Path(PROJECT_DIR) / 'seq2seq_layer_table.csv', index=False)
table

append_section(
    NOTES_PATH, 'Primary model architecture',
    f'Parameter counts: {param_counts(model)}\n\n' + table.to_markdown(index=False),
)

## 6. Training (Task 6)
Cross-entropy with padding ignored, gradient clipping, checkpointing on best val loss, early stopping.

In [ ]:
from q2_nmt_rnn.src.training.engine import fit

result = fit(
    model, train_loader, val_loader, lr=1e-3, max_epochs=40, patience=6, device=device,
    grad_clip_norm=1.0, checkpoint_path=Path(PROJECT_DIR) / 'checkpoints' / 'seq2seq_rnn.pth',
)
result.history.to_csv(Path(PROJECT_DIR) / 'training_history.csv', index=False)
print('best_val_loss:', result.best_val_loss, 'epochs_trained:', result.epochs_trained)
result.history.plot(x='epoch', y=['train_loss', 'val_loss'], title='Seq2Seq convergence')

append_section(
    NOTES_PATH, 'Training',
    f'Epochs trained: {result.epochs_trained}, best val loss: {result.best_val_loss:.4f}\n\n'
    + result.history.to_markdown(index=False),
)

## 7. Qualitative sanity check
Not an assignment task on its own -- just a quick look at what the trained model produces, via greedy (argmax) decoding, to eyeball alongside the loss curve above.

In [ ]:
@torch.no_grad()
def greedy_translate(model, sentence: str, max_len: int = 50) -> str:
    from q2_nmt_rnn.src.config import BOS_IDX, EOS_IDX

    model.eval()
    src_ids = torch.tensor([src_vocab.encode(tokenize(sentence))], device=device)
    hidden = model.encoder(src_ids)

    generated = [BOS_IDX]
    for _ in range(max_len):
        step_input = torch.tensor([[generated[-1]]], device=device)
        output, hidden = model.decoder.rnn(model.decoder.embedding(step_input), hidden)
        next_id = model.decoder.output_proj(output)[0, -1].argmax().item()
        if next_id == EOS_IDX:
            break
        generated.append(next_id)

    return ' '.join(tgt_vocab.decode(generated[1:]))

for sentence in splits['test']['en'].head(5):
    print(sentence, '->', greedy_translate(model, sentence))

samples_md = '\n'.join(
    f'- **{sentence}** -> {greedy_translate(model, sentence)}' for sentence in splits['test']['en'].head(5)
)
append_section(NOTES_PATH, 'Qualitative translation samples', samples_md)